In [ ]:
from pathlib import Path
import pandas as pd
import pm4py

DATA_DIR = Path("./../MIMICEL_data")

SPLITS = {
    "train": DATA_DIR / "mimicel_train.csv",
    "val": DATA_DIR / "mimicel_val.csv",
    "test": DATA_DIR / "mimicel_test.csv",
}

CASE_COL = "stay_id"
TIME_COL = "timestamps"
ACT_COL = "activity"

for split, csv_path in SPLITS.items():
    print(f"[LOAD] {split}: {csv_path}")

    df = pd.read_csv(csv_path, low_memory=False)

    TIME_COL_USED = TIME_COL if TIME_COL in df.columns else "timestamp"

    df[TIME_COL_USED] = pd.to_datetime(df[TIME_COL_USED], errors="coerce")
    df = df.dropna(subset=[CASE_COL, TIME_COL_USED, ACT_COL])

    df[CASE_COL] = df[CASE_COL].astype(str)
    df[ACT_COL] = df[ACT_COL].astype(str)

    df = df.sort_values([CASE_COL, TIME_COL_USED]).reset_index(drop=True)

    df = df.rename(columns={
        CASE_COL: "case:concept:name",
        TIME_COL_USED: "time:timestamp",
        ACT_COL: "concept:name"
    })

    df["case:concept:name"] = df["case:concept:name"].astype(str)
    df["concept:name"] = df["concept:name"].astype(str)

    event_log = pm4py.convert_to_event_log(
        df,
        case_id_key="case:concept:name"
    )

    xes_path = DATA_DIR / f"mimicel_{split}.xes"
    pm4py.write_xes(event_log, str(xes_path))

    print(f"[SAVE] {xes_path}")
    print("cases:", df["case:concept:name"].nunique())
    print("events:", len(df))

[LOAD] train: ..\MIMICEL_data\mimicel_train.csv
